# 06 실전 반도체 공정 데이터분석 강의자료

원본: `D:/Githubs/DataAnaysis_WS/kitae-data/Web/실전_반도체_공정_데이터분석_강의자료.html`

이 노트북은 HTML 강의자료에 들어 있던 프로그램 소스를 실행 가능한 코드 셀로 옮긴 버전입니다. 각 코드 셀 앞에는 수업 흐름을 따라갈 수 있도록 간단한 설명을 붙였습니다.

## 1단계 · 라이브러리 불러오기

### 1단계 · 라이브러리 불러오기
- 원본 코드: `Python`
- 설명: 필요한 행과 열만 골라 보며 DataFrame 선택 문법을 연습합니다.
- 수업 메모: 이전 시간 + SelectKBest 만 추가

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif   # ⭐ NEW
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc

plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

## 2단계 · 데이터 불러오기 & 큰 테이블 다루기

### 2단계 · 데이터 불러오기 & 큰 테이블 다루기
- 원본 코드: `Python — 데이터 불러오기 & 기본 탐색`
- 설명: 컬럼이 590개나 되니 요약 정보 로 살핍시다

In [ ]:
df = pd.read_csv('fab.csv')

print(f"📊 데이터 크기: {df.shape[0]}행 × {df.shape[1]}열")
print(f"📋 첫 5컬럼: {df.columns[:5].tolist()}")
print(f"📋 마지막 컬럼: {df.columns[-3:].tolist()}")

# 컬럼이 너무 많으니 일부만 미리보기
df.iloc[:, :6].head()

# 590개 컬럼을 다 출력하지 않게 verbose=False
df.info(verbose=False)

# 일부 센서만 통계
df[['Sensor0', 'Sensor1', 'Sensor2']].describe().round(2)

# 라벨 분포 (불균형 확인!)
df['Pass_Fail'].value_counts()

## 3단계 · 결측값 처리 (적극 모드)

### 3단계 · 결측값 처리 (적극 모드)
- 원본 코드: `Python — 결측 컬럼 제거 + 중앙값 채우기`
- 설명: 결측이 너무 많은 컬럼은 통째로 제거

In [ ]:
# 1️⃣ 컬럼별 결측률 계산
miss_pct = (df.isnull().sum() / len(df) * 100).round(2)
print(miss_pct.sort_values(ascending=False).head(10))

# 2️⃣ 결측률 50% 초과 컬럼 제거
THRESHOLD = 50.0
cols_to_drop = miss_pct[miss_pct > THRESHOLD].index.tolist()
print(f"🗑️ 제거할 컬럼: {len(cols_to_drop)}개")
df = df.drop(columns=cols_to_drop)

# 3️⃣ 남은 결측치는 중앙값으로 채우기 (타겟 제외!)
numeric_cols = df.select_dtypes(include='number').columns.tolist()
numeric_cols.remove('Pass_Fail')   # ⚠️ 타겟은 절대 채우지 말 것!

df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())
print(f"✅ 결측 처리 완료. 남은 결측: {df.isnull().sum().sum()}")

## 4단계 · 의미 없는 컬럼 제거 NEW

### 4단계 · 의미 없는 컬럼 제거 NEW
- 원본 코드: `Python — 분산 0 컬럼 제거`
- 설명: 분산이 거의 없는 컬럼을 제거해 학습에 불필요한 변수를 줄입니다.
- 수업 메모: 분산이 0인 센서는 학습에 전혀 도움 안 됩니다

In [ ]:
# 분산 = 0 컬럼 + 거의 0인 컬럼 모두 제거
variances = df[numeric_cols].var()

constant_cols    = variances[variances == 0].index.tolist()
near_constant    = variances[(variances > 0) & (variances < 1e-6)].index.tolist()

print(f"분산 0  컬럼: {len(constant_cols)}개")
print(f"분산 ≈0 컬럼: {len(near_constant)}개")

to_drop_var = constant_cols + near_constant
df = df.drop(columns=to_drop_var)
numeric_cols = [c for c in numeric_cols if c not in to_drop_var]

print(f"✅ 사용 가능한 센서: {len(numeric_cols)}개")

## 5단계 · 특성 선택 NEW

### 5단계 · 특성 선택 NEW
- 원본 코드: `Python — 상위 K개 센서 자동 선택`
- 설명: 통계 검정으로 예측에 중요한 상위 센서 컬럼을 선택합니다.
- 수업 메모: 중요한 K개 센서만 추리기 (ANOVA F-검정)

In [ ]:
# 타겟을 0/1로 변환: 불량(1)을 양성 클래스로
y = (df['Pass_Fail'] == 1).astype(int)
X_all = df[numeric_cols].copy()

# 상위 K=20 개 센서 자동 선택
K = 20
selector = SelectKBest(score_func=f_classif, k=K)
selector.fit(X_all, y)

f_scores   = pd.Series(selector.scores_, index=numeric_cols)\
               .replace([np.inf, -np.inf], np.nan).dropna()
top_k_cols = f_scores.sort_values(ascending=False).head(K).index.tolist()

print(f"🏆 선택된 상위 {K}개 센서:")
for i, col in enumerate(top_k_cols, 1):
    print(f"  {i:2d}. {col}  F={f_scores[col]:6.1f}")

### 5단계 · 특성 선택 NEW
- 원본 코드: `Python — 상위 K 시각화`
- 설명: 통계 검정으로 예측에 중요한 상위 센서 컬럼을 선택합니다.
- 수업 메모: 중요한 K개 센서만 추리기 (ANOVA F-검정)

In [ ]:
top = f_scores.sort_values(ascending=False).head(K)
plt.figure(figsize=(10, 7))
plt.barh(top.index[::-1], top.values[::-1], color='#3498db')
plt.xlabel('F-값 (클수록 유용)')
plt.title('상위 K개 센서 F-점수')
plt.tight_layout()
plt.show()

## 6단계 · 핵심 변수 EDA

### 6단계 · 핵심 변수 EDA
- 원본 코드: `Python — 상위 6개 박스플롯`
- 설명: 통계 검정으로 예측에 중요한 상위 센서 컬럼을 선택합니다.
- 수업 메모: 상위 K개로 줄였으니 이제 시각화 가능!

In [ ]:
top6 = top_k_cols[:6]
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

for i, col in enumerate(top6):
    sns.boxplot(x='Pass_Fail', y=col, data=df,
                hue='Pass_Fail',
                palette={-1: '#2ecc71', 1: '#e74c3c'},
                legend=False, ax=axes[i], width=0.5)
    axes[i].set_title(f'{col}  (F={f_scores[col]:.1f})')
    axes[i].set_xticklabels(['정상(-1)', '불량(+1)'])

plt.suptitle('📦 상위 6개 센서 — 정상 vs 불량 분포')
plt.tight_layout()
plt.show()

### 6단계 · 핵심 변수 EDA
- 원본 코드: `Python — 상위 K 상관관계 히트맵`
- 설명: 통계 검정으로 예측에 중요한 상위 센서 컬럼을 선택합니다.
- 수업 메모: 상위 K개로 줄였으니 이제 시각화 가능!

In [ ]:
corr_top = df[top_k_cols + ['Pass_Fail']].corr().round(2)

plt.figure(figsize=(13, 10))
mask = np.triu(np.ones_like(corr_top, dtype=bool))
sns.heatmap(corr_top, annot=True, fmt='.2f',
            cmap='RdYlGn', center=0, mask=mask,
            linewidths=0.5)
plt.title('🔗 상위 센서 간 상관관계 + Pass_Fail')
plt.show()

## 7단계 · 데이터 전처리 & 분리

### 7단계 · 데이터 전처리 & 분리
- 원본 코드: `Python — 분리 & 스케일링`
- 설명: 통계 검정으로 예측에 중요한 상위 센서 컬럼을 선택합니다.
- 수업 메모: 상위 K개 센서만 입력으로 사용

In [ ]:
X = df[top_k_cols].copy()    # 상위 K개 센서만!
# y = (df['Pass_Fail'] == 1).astype(int)  # 위에서 이미 만듦

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y          # ⭐ 불균형 비율 유지 필수!
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f"학습: {X_train.shape}, 그 중 불량 {y_train.sum()}건")
print(f"테스트: {X_test.shape}, 그 중 불량 {y_test.sum()}건")

## 8단계 · 머신러닝 모델 학습

### 8단계 · 머신러닝 모델 학습
- 원본 코드: `Python — 모델 학습 (불균형 처리)`
- 설명: 전처리한 데이터로 머신러닝 모델을 학습하고 예측합니다.
- 수업 메모: class_weight='balanced' 한 줄로 불균형 해결

In [ ]:
# 1) 로지스틱 회귀
lr_model = LogisticRegression(
    random_state=42, max_iter=1000,
    class_weight='balanced'   # ⭐ 소수 클래스에 가중치
)
lr_model.fit(X_train_scaled, y_train)

# 2) 랜덤 포레스트
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    random_state=42,
    class_weight='balanced'
)
rf_model.fit(X_train_scaled, y_train)

# 예측
y_pred_lr = lr_model.predict(X_test_scaled)
y_pred_rf = rf_model.predict(X_test_scaled)

## 9단계 · 모델 평가 — Recall 최우선

### 9단계 · 모델 평가 — Recall 최우선
- 원본 코드: `Python — 혼동행렬 + 분류 리포트`
- 설명: 전처리한 데이터로 머신러닝 모델을 학습하고 예측합니다.
- 수업 메모: 불균형 데이터에서는 정확도가 거짓말을 합니다

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (name, model) in zip(axes,
        [('로지스틱 회귀', lr_model), ('랜덤 포레스트', rf_model)]):
    cm = confusion_matrix(y_test, model.predict(X_test_scaled))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['정상(0)', '불량(1)'],
                yticklabels=['정상(0)', '불량(1)'])
    ax.set_title(name)
plt.show()

# 상세 리포트
print(classification_report(y_test, lr_model.predict(X_test_scaled),
      target_names=['정상(0)', '불량(1)']))

### 9단계 · 모델 평가 — Recall 최우선
- 원본 코드: `Python — ROC 커브 비교`
- 설명: 전처리한 데이터로 머신러닝 모델을 학습하고 예측합니다.
- 수업 메모: 불균형 데이터에서는 정확도가 거짓말을 합니다

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
for name, model, color in [
        ('로지스틱 회귀', lr_model, '#3498db'),
        ('랜덤 포레스트', rf_model, '#e74c3c')]:
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, linewidth=2.5,
            label=f'{name} (AUC={roc_auc:.3f})')
ax.plot([0,1], [0,1], 'k--', alpha=0.5)
ax.set_xlabel('FPR'); ax.set_ylabel('TPR (= 불량 Recall)')
ax.legend(); plt.show()

### 9단계 · 모델 평가 — Recall 최우선
- 원본 코드: `Python — 특성 중요도`
- 설명: 전처리한 데이터로 머신러닝 모델을 학습하고 예측합니다.
- 수업 메모: 불균형 데이터에서는 정확도가 거짓말을 합니다

In [ ]:
imp = pd.Series(rf_model.feature_importances_,
                index=top_k_cols).sort_values()

plt.figure(figsize=(10, 7))
plt.barh(imp.index, imp.values, color='#e74c3c')
plt.xlabel('Feature Importance')
plt.title('🌲 랜덤 포레스트 특성 중요도')
plt.show()

## 초보자 필수 보강 실습

모델을 만드는 것보다 중요한 것은 **무엇을 맞혀야 하는지**, **어떤 실수를 줄여야 하는지**를
이해하는 일입니다. 반도체 불량 탐지에서는 불량을 정상으로 놓치는 `False Negative`와
불량 재현율(`Recall`)을 특히 주의해서 봅니다.


### 실습 1 · 클래스 불균형과 기준 모델 이해하기

**목표:** 항상 정상이라고 예측하는 단순 모델과 비교해 정확도의 함정을 확인합니다.

아래 코드 셀의 주석을 보고 먼저 직접 작성해 보세요. 막히면 힌트를 확인하고,
마지막에 정답 예제를 실행해 결과를 비교합니다.


In [ ]:
# TODO 1: y_test의 정상/불량 건수와 비율을 출력하세요.
# TODO 2: 항상 정상(0)만 예측하는 배열을 만드세요.
# TODO 3: 이 기준 모델의 정확도와 불량 Recall을 계산하세요.


<details>
<summary><strong>힌트 보기</strong></summary>

`value_counts(normalize=True)`와 `accuracy_score`, `recall_score`를 사용합니다.

</details>

**정답 예제:** 먼저 직접 시도한 뒤 아래 셀을 실행하세요.


In [ ]:
from sklearn.metrics import accuracy_score, recall_score

print('테스트 판정 건수:')
print(y_test.value_counts().sort_index())
print('\n테스트 판정 비율(%):')
print(y_test.value_counts(normalize=True).sort_index().mul(100).round(2))

always_normal = np.zeros(len(y_test), dtype=int)
baseline_accuracy = accuracy_score(y_test, always_normal)
baseline_recall = recall_score(y_test, always_normal, zero_division=0)
print(f'항상 정상 모델 정확도: {baseline_accuracy:.3f}')
print(f'항상 정상 모델 불량 Recall: {baseline_recall:.3f}')
print('정확도가 높아도 불량 Recall이 0이면 불량 탐지 모델로 쓸 수 없습니다.')


### 실습 2 · 혼동행렬을 숫자로 읽기

**목표:** 정상/불량 예측의 네 가지 경우와 놓친 불량 수를 직접 확인합니다.

아래 코드 셀의 주석을 보고 먼저 직접 작성해 보세요. 막히면 힌트를 확인하고,
마지막에 정답 예제를 실행해 결과를 비교합니다.


In [ ]:
# TODO 1: 랜덤 포레스트 예측 결과로 혼동행렬을 만드세요.
# TODO 2: tn, fp, fn, tp 네 값으로 나누어 저장하세요.
# TODO 3: 불량 Recall = tp / (tp + fn)을 직접 계산하세요.


<details>
<summary><strong>힌트 보기</strong></summary>

2x2 혼동행렬에는 `ravel()`을 적용할 수 있습니다. 분모가 0인지도 확인하세요.

</details>

**정답 예제:** 먼저 직접 시도한 뒤 아래 셀을 실행하세요.


In [ ]:
rf_predictions = rf_model.predict(X_test_scaled)
tn, fp, fn, tp = confusion_matrix(y_test, rf_predictions).ravel()
manual_recall = tp / (tp + fn) if (tp + fn) else 0

print(f'TN(정상을 정상): {tn}')
print(f'FP(정상을 불량): {fp}')
print(f'FN(불량을 정상으로 놓침): {fn}')
print(f'TP(불량을 불량): {tp}')
print(f'불량 Recall: {manual_recall:.3f}')


### 실습 3 · 예측 임계값과 Recall의 관계

**목표:** 불량 판정 기준을 바꿀 때 Recall과 Precision이 어떻게 달라지는지 비교합니다.

아래 코드 셀의 주석을 보고 먼저 직접 작성해 보세요. 막히면 힌트를 확인하고,
마지막에 정답 예제를 실행해 결과를 비교합니다.


In [ ]:
# TODO 1: 랜덤 포레스트의 불량 확률을 구하세요.
# TODO 2: 임계값 0.3, 0.5, 0.7마다 예측값을 만드세요.
# TODO 3: 각 임계값의 Precision과 Recall을 표로 정리하세요.


<details>
<summary><strong>힌트 보기</strong></summary>

확률이 임계값 이상이면 1로 바꾸고 `precision_score`, `recall_score`를 계산합니다.

</details>

**정답 예제:** 먼저 직접 시도한 뒤 아래 셀을 실행하세요.


In [ ]:
from sklearn.metrics import precision_score, recall_score

rf_probabilities = rf_model.predict_proba(X_test_scaled)[:, 1]
threshold_rows = []
for threshold in [0.3, 0.5, 0.7]:
    threshold_prediction = (rf_probabilities >= threshold).astype(int)
    threshold_rows.append({
        '임계값': threshold,
        'Precision': precision_score(y_test, threshold_prediction, zero_division=0),
        'Recall': recall_score(y_test, threshold_prediction, zero_division=0),
        '불량예측수': int(threshold_prediction.sum())
    })

threshold_table = pd.DataFrame(threshold_rows).round(3)
threshold_table


### 실습 4 · 두 모델을 같은 기준으로 비교하기

**목표:** 정확도만 보지 않고 불량 Precision, Recall, F1을 한 표에서 비교합니다.

아래 코드 셀의 주석을 보고 먼저 직접 작성해 보세요. 막히면 힌트를 확인하고,
마지막에 정답 예제를 실행해 결과를 비교합니다.


In [ ]:
# TODO 1: 로지스틱 회귀와 랜덤 포레스트의 예측값을 만드세요.
# TODO 2: 각 모델의 Accuracy, Precision, Recall, F1을 계산하세요.
# TODO 3: Recall이 높은 순서로 정렬하세요.


<details>
<summary><strong>힌트 보기</strong></summary>

여러 모델을 리스트에 담아 반복하면 같은 계산 기준을 적용할 수 있습니다.

</details>

**정답 예제:** 먼저 직접 시도한 뒤 아래 셀을 실행하세요.


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

comparison_rows = []
for model_name, model in [
    ('로지스틱 회귀', lr_model),
    ('랜덤 포레스트', rf_model)
]:
    prediction = model.predict(X_test_scaled)
    comparison_rows.append({
        '모델': model_name,
        'Accuracy': accuracy_score(y_test, prediction),
        'Precision_불량': precision_score(y_test, prediction, zero_division=0),
        'Recall_불량': recall_score(y_test, prediction, zero_division=0),
        'F1_불량': f1_score(y_test, prediction, zero_division=0)
    })

model_comparison = (
    pd.DataFrame(comparison_rows)
    .sort_values('Recall_불량', ascending=False)
    .round(3)
)
model_comparison


### 실습 5 · 놓친 불량 사례 살펴보기

**목표:** False Negative 행을 찾아 어떤 센서값을 추가로 점검할지 준비합니다.

아래 코드 셀의 주석을 보고 먼저 직접 작성해 보세요. 막히면 힌트를 확인하고,
마지막에 정답 예제를 실행해 결과를 비교합니다.


In [ ]:
# TODO 1: y_test가 1이면서 랜덤 포레스트 예측이 0인 위치를 찾으세요.
# TODO 2: 원래 X_test에서 해당 행을 선택하세요.
# TODO 3: 놓친 불량 수와 센서값 일부를 출력하세요.


<details>
<summary><strong>힌트 보기</strong></summary>

`(y_test == 1) & (예측 == 0)` 조건을 만들고 불리언 배열의 위치로 `X_test`를 선택합니다.

</details>

**정답 예제:** 먼저 직접 시도한 뒤 아래 셀을 실행하세요.


In [ ]:
false_negative_mask = (y_test.to_numpy() == 1) & (rf_predictions == 0)
false_negative_cases = X_test.loc[false_negative_mask].copy()

print(f'랜덤 포레스트가 놓친 불량: {len(false_negative_cases)}건')
display_cols = top_k_cols[:5]
if len(false_negative_cases) > 0:
    display(false_negative_cases[display_cols].head())
    print('이 행들은 원인 확정 대상이 아니라 추가 공정 점검 대상입니다.')
else:
    print('현재 테스트 데이터에서는 놓친 불량이 없습니다.')


### 꼭 알아둘 데이터 누수

실제 프로젝트에서는 **데이터를 학습용과 평가용으로 먼저 분리한 뒤** 다음 작업을
학습 데이터에만 맞춰야 합니다.

- 결측값을 채울 중앙값 계산
- 중요한 센서 선택
- 표준화 평균과 표준편차 계산

평가 데이터의 정보를 미리 사용하면 모델 성능이 실제보다 좋아 보일 수 있습니다.
AI에게 프로그램을 요청할 때도 이 순서를 명확히 적어 주는 것이 좋습니다.


## AI와 함께 만드는 미니 프로젝트

### 프로젝트: 불량 탐지 모델과 평가 리포트

아래 프롬프트를 AI 도구에 붙여 넣으면 이번 노트북에서 배운 범위로 실제 프로그램을
만들어 볼 수 있습니다. AI가 만든 코드를 한꺼번에 실행하지 말고, 셀별로 읽고 실행하며
자신의 데이터 열 이름과 결과를 확인하세요.

**프롬프트 사용 설명**

이 프롬프트의 핵심은 AI에게 모델 종류보다 평가 목적과 데이터 누수 방지 순서를 명확히 알려 주는 것입니다. 생성된 코드는 먼저 작은 셀로 나누어 실행하고, 테스트 데이터가 전처리 학습에 사용되지 않았는지 확인하세요.


### 복사해서 사용할 프롬프트

```text
당신은 초보자에게 코드의 이유를 설명하는 반도체 데이터 분석가입니다.

[역할]
fab.csv로 정상/불량을 분류하고 불량을 놓치지 않는 데 초점을 둔 주피터 노트북 프로그램을 작성하세요.

[입력 데이터]
- 파일명: fab.csv
- 타겟 열: Pass_Fail (-1=정상, 1=불량)
- 나머지 숫자 열은 센서값이며 결측값과 상수 열이 포함될 수 있습니다.

[구현 요구사항]
1. 클래스 건수와 비율을 확인하고 정확도만 보면 안 되는 이유를 설명하세요.
2. 데이터를 학습 80%, 테스트 20%로 stratify 분리하세요.
3. 결측률 기준 열 제거, 중앙값 채우기, 상수 열 제거, SelectKBest, StandardScaler는
   반드시 학습 데이터에만 fit하고 테스트 데이터에는 transform만 하세요.
4. 로지스틱 회귀와 랜덤 포레스트에 class_weight='balanced'를 적용하세요.
5. Accuracy, 불량 Precision, 불량 Recall, 불량 F1, ROC-AUC를 같은 표로 비교하세요.
6. 혼동행렬을 그리고 False Negative 수를 눈에 띄게 출력하세요.
7. 기본 임계값 0.5에서 결과를 평가하고, 0.3과 0.7은 학습용 비교표로만 보여주세요.
8. random_state=42를 사용하고 각 단계 앞에 초보자용 설명을 작성하세요.

[결과물]
- 위에서 아래로 실행 가능한 주피터 노트북 코드
- model_comparison.csv
- 모델별 혼동행렬과 ROC 곡선
- 놓친 불량 사례의 상위 센서값 표
- 어떤 모델을 선택할지 근거가 포함된 3문장 결론

[검증]
- 학습/테스트 행 수와 불량 건수를 출력하세요.
- 전처리 객체가 테스트 데이터에 fit되지 않았음을 코드 주석으로 확인하세요.
- 예측값 개수가 y_test 행 수와 같은지 assert로 검사하세요.
- 결과를 '원인 규명'으로 과장하지 말고 추가 공정 검토가 필요하다고 표시하세요.
```


### 결과 검토 체크리스트

AI가 프로그램을 작성했다고 해서 결과가 자동으로 맞는 것은 아닙니다.

- [ ] CSV 파일 경로와 열 이름이 실제 데이터와 같은가?
- [ ] 원본 데이터를 복사한 뒤 정리했는가?
- [ ] 처리 전후 행 수와 결측값 수가 설명 가능한가?
- [ ] 그래프의 축, 단위, 범례가 데이터 의미와 맞는가?
- [ ] 합격/불합격 숫자의 의미를 반대로 사용하지 않았는가?
- [ ] 오류 메시지가 나오면 해당 셀만 읽고 원인을 설명할 수 있는가?
- [ ] 분석 결과를 공정 원인으로 단정하지 않았는가?
- [ ] 실제 회사 데이터라면 보안 규정을 지켰는가?

마지막으로 AI에게 다음과 같이 요청해 보세요.

> 위 코드를 한 셀씩 설명하고, 각 셀에서 초보자가 확인해야 할 출력값을 알려줘.


## AI로 학습 내용을 확인하고 확장하기

AI는 정답을 대신 제출하는 도구보다 **내 설명의 빈틈을 찾고, 코드를 검토하고,
새로운 문제를 설계하는 학습 파트너**로 사용할 때 효과적입니다.

아래 프롬프트는 특정 서비스에 종속되지 않습니다. 대괄호나 `여기에 ...`라고 적힌 부분만
자신의 상황에 맞게 바꿔 사용하세요. 실제 회사 데이터, 장비명, 레시피값, Lot 식별자는
외부 AI 서비스에 직접 붙여 넣지 않습니다.


### 1. AI 튜터로 학습 내용 확인하기

```text
나는 반도체 불량 분류 모델을 복습하고 있습니다. 다음 주제에서 확인 문제를 6개 내주세요:
클래스 불균형, stratify 분할, fit과 transform, 데이터 누수, 혼동행렬,
불량 Precision·Recall, 예측 임계값.

한 번에 한 문제만 내고 내가 답할 때까지 정답을 공개하지 마세요.
계산 문제에는 작은 혼동행렬 숫자를 사용하고, 코드 문제에는 10줄 이하 예제를 사용하세요.
답을 평가할 때 불량을 정상으로 놓치는 False Negative 관점도 설명하세요.
마지막에는 실무에서 반드시 확인할 항목 5개를 체크리스트로 정리하세요.
```


**사용 방법:** 먼저 노트북을 보지 않고 답한 뒤, AI의 설명을 원래 강의 코드와 비교합니다.
AI가 제시한 함수가 실제 데이터 열과 맞는지 작은 가상 데이터로 직접 실행해 확인하세요.


### 2. 작성한 프로그램 개선하기

```text
아래 반도체 불량 분류 코드를 데이터 누수와 평가 오류 중심으로 검토하세요.

[필수 확인]
- Pass_Fail의 -1=정상, 1=불량 매핑
- train_test_split과 stratify 사용
- 결측 처리, 특성 선택, StandardScaler가 학습 데이터에만 fit되는지
- class_weight='balanced'와 random_state 설정
- Accuracy 외에 불량 Precision, Recall, F1, ROC-AUC를 계산하는지
- 혼동행렬의 False Negative를 정확히 읽는지

[응답 형식]
1. 심각도별 문제 목록
2. 데이터 누수를 막은 최소 수정 코드
3. 수정 전후 처리 순서 비교
4. 예측 개수, 클래스 분포, 혼동행렬 합계를 확인하는 assert 코드

새로운 고급 모델이나 복잡한 튜닝은 추가하지 마세요.

[검토할 코드]
여기에 코드를 붙여 넣으세요.
```


**개선 결과 확인 순서**

1. AI가 바꾼 줄과 이유를 먼저 읽습니다.
2. 원본 코드는 남겨 두고 복사본에서 수정 코드를 실행합니다.
3. 행 수, 결측값 수, 타겟 분포처럼 바뀌면 안 되는 값을 비교합니다.
4. 결과가 달라졌다면 오류 수정 때문인지 분석 의미가 바뀐 것인지 확인합니다.
5. 이해하지 못한 고급 문법은 쉬운 코드로 다시 작성해 달라고 요청합니다.


## AI 프롬프트로 도전하는 추가 미니 프로젝트

아래 두 프로젝트는 새 라이브러리를 많이 배우기보다 현재 강의의 기능을 다른 문제에
조합하는 연습입니다. AI가 만든 첫 답을 완성본으로 보지 말고, 입력 열·중간 출력·저장 파일을
하나씩 확인하며 개선하세요.


### 추가 프로젝트 1 · 불량 판정 임계값 비교 도구

```text
학습된 분류 모델의 불량 확률을 사용해 임계값 0.2부터 0.8까지 0.1 간격으로 비교하는 코드를 작성하세요.
각 임계값의 Precision, Recall, F1, False Positive, False Negative, 불량 예측 수를 표로 만드세요.
Recall과 Precision 변화를 선 그래프로 그리고 False Negative가 가장 적은 임계값도 표시하세요.
임계값 선택은 테스트 데이터가 아니라 별도 검증 데이터에서 해야 한다는 설명을 포함하세요.
초보자가 기존 y_valid와 y_valid_prob 변수에 연결할 수 있도록 독립적인 코드 셀로 작성하세요.
```


### 추가 프로젝트 2 · 놓친 불량 사례 점검 도구

```text
X_test, y_test, y_pred와 센서 열 이름을 이용해 False Negative 사례를 점검하는 프로그램을 작성하세요.
놓친 불량 행을 찾고, True Positive 불량과 정상 데이터의 센서 중앙값을 함께 비교하세요.
차이가 큰 센서 상위 10개를 표로 만들고 놓친 불량의 센서값을 CSV로 저장하세요.
표본 수가 0일 때도 오류 없이 안내하고, 결과를 불량 원인으로 단정하지 마세요.
마지막에 공정 엔지니어에게 확인할 질문 3개를 자동으로 출력하세요.
```


### AI 결과 검증과 보안 체크

- [ ] 가상 데이터나 비식별 데이터로 먼저 실행했는가?
- [ ] 사용한 열 이름과 합격·불합격 값의 의미가 맞는가?
- [ ] 원본 데이터와 코드의 복사본을 보존했는가?
- [ ] 처리 전후 행 수, 결측값 수, 클래스 분포를 비교했는가?
- [ ] 상관관계나 모델 중요도를 공정 원인으로 단정하지 않았는가?
- [ ] 회사 보안 규정상 외부 입력이 금지된 정보를 제거했는가?

AI의 설명과 실행 결과가 다르면 실행 결과를 우선하고, 오류 메시지와 최소 예제만 제공해
다시 질문하세요.


## 마무리

위 코드 셀을 순서대로 실행하면서 데이터 불러오기, 탐색, 정리, 시각화, 모델링 흐름을 복습해 보세요. 일부 셀은 앞에서 만든 변수(`df`, `df_clean`, `top_k_cols` 등)를 이어서 사용합니다.